In [3]:
%pip install beautifulsoup4 requests nltk sentence-transformers chromadb pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ============================================
# WEEK 1 NLP IMPLEMENTATION PROJECT
# Web Scraping + Tokenization + Embeddings + ChromaDB
# ============================================

# ============================================
# INSTALL REQUIRED LIBRARIES
# Run separately if needed:
#
# pip install beautifulsoup4 requests nltk
# pip install sentence-transformers chromadb pandas numpy
# ============================================


# ============================================
# IMPORTS
# ============================================

import csv
import json
import requests
import pandas as pd
import nltk
import numpy as np

from bs4 import BeautifulSoup

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

from sentence_transformers import SentenceTransformer

import chromadb


# ============================================
# DOWNLOAD NLTK DATA
# ============================================

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


# ============================================
# STEP 1 — WEB SCRAPING
# ============================================

print("\nSTEP 1 — SCRAPING NPR NEWS ARTICLES\n")

base_url = "https://www.npr.org"

url = "https://www.npr.org/sections/news/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    headers=headers,
    timeout=10
)

soup = BeautifulSoup(response.text, "html.parser")


# ============================================
# GET ARTICLE LINKS
# ============================================

links = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    # NPR article links usually contain year/date
    if "202" in href and "/202" in href:

        if href not in links:

            links.append(href)

print("Total Links Found:", len(links))


# ============================================
# SCRAPE ARTICLE CONTENT
# ============================================

articles = []

for link in links[:20]:

    try:

        article_response = requests.get(
            link,
            headers=headers,
            timeout=10
        )

        if article_response.status_code != 200:
            continue

        article_soup = BeautifulSoup(
            article_response.text,
            "html.parser"
        )

        # ------------------------------------
        # TITLE
        # ------------------------------------

        title_tag = article_soup.find("h1")

        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)

        # Clean title
        title = title.replace("\n", " ")
        title = title.replace("\r", " ")
        title = title.replace(",", " ")

        # ------------------------------------
        # CONTENT
        # ------------------------------------

        paragraphs = article_soup.find_all("p")

        content = ""

        for p in paragraphs:

            text = p.get_text(strip=True)

            # Clean text
            text = text.replace("\n", " ")
            text = text.replace("\r", " ")
            text = text.replace(",", " ")
            text = text.replace('"', "")

            if len(text) > 40:

                content += text + " "

        # Remove extra spaces
        content = " ".join(content.split())

        # ------------------------------------
        # STORE ARTICLE
        # ------------------------------------

        if len(content.split()) > 100:

            articles.append({
                "title": title,
                "content": content,
                "url": link
            })

            print(f"Scraped: {title}")

    except Exception as e:

        print("Error:", e)

print("\nTotal Articles Scraped:", len(articles))


# ============================================
# PREVIEW ARTICLES
# ============================================

print("\nSAMPLE ARTICLES\n")

for i, article in enumerate(articles[:5]):

    print(f"{i+1}. {article['title']}\n")


# ============================================
# STEP 2 — SAVE FULL CORPUS JSON
# ============================================

print("\nSTEP 2 — SAVING CORPUS\n")

with open(
    "corpus.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        articles,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Corpus saved as corpus.json")


# ============================================
# CREATE DATAFRAME
# ============================================

df = pd.DataFrame(articles)

print("\nDATAFRAME PREVIEW\n")

print(df.head())


# ============================================
# CREATE SHORT CSV VERSION
# ============================================

# df["content_preview"] = df["content"].str[:300]

df_export = df[
    ["title", "content", "url"]
].iloc[:-1]


# ============================================
# EXPORT CSV
# ============================================

df_export.to_csv(
    "news_corpus.csv",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL
)

print("\nCSV exported successfully.")


# ============================================
# STEP 3 — TOKENIZATION
# ============================================

print("\nSTEP 3 — TOKENIZATION\n")

if len(articles) == 0:

    print("No articles scraped.")
    exit()

sample_text = articles[0]["content"]

print("Sample Text:\n")

print(sample_text[:1000])


# ============================================
# WORD TOKENIZATION
# ============================================

word_tokens = word_tokenize(sample_text)

print("\nWORD TOKENS:\n")

print(word_tokens[:50])


# ============================================
# SENTENCE TOKENIZATION
# ============================================

sentence_tokens = sent_tokenize(sample_text)

print("\nSENTENCE TOKENS:\n")

print(sentence_tokens[:5])


# ============================================
# STOPWORD REMOVAL
# ============================================

stop_words = set(stopwords.words('english'))

clean_tokens = []

for word in word_tokens:

    if (
        word.lower() not in stop_words
        and word.isalpha()
    ):

        clean_tokens.append(word)

print("\nTOKENS AFTER STOPWORD REMOVAL:\n")

print(clean_tokens[:50])


# ============================================
# STEP 4 — GENERATE EMBEDDINGS
# ============================================

print("\nSTEP 4 — GENERATING EMBEDDINGS\n")

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

texts = [
    article["content"]
    for article in articles
]

embeddings = model.encode(texts)

print("Embedding Shape:", embeddings.shape)


# ============================================
# SAVE EMBEDDINGS
# ============================================

np.save(
    "embeddings.npy",
    embeddings
)

print("Embeddings saved as embeddings.npy")


# ============================================
# VIEW SAMPLE EMBEDDING
# ============================================

print("\nSAMPLE EMBEDDING VECTOR\n")

print(embeddings[0][:20])


# ============================================
# STEP 5 — CHROMADB VECTOR DATABASE
# ============================================

print("\nSTEP 5 — STORING IN CHROMADB\n")

client = chromadb.Client()

collection = client.get_or_create_collection(
    "news_articles"
)


# ============================================
# STORE DOCUMENTS
# ============================================

collection.add(
    documents=texts,

    embeddings=embeddings.tolist(),

    ids=[
        str(i)
        for i in range(len(texts))
    ],

    metadatas=[
        {
            "title": article["title"],
            "url": article["url"]
        }
        for article in articles
    ]
)

print("Documents stored successfully.")


# ============================================
# STEP 6 — SEMANTIC SEARCH
# ============================================

print("\nSTEP 6 — SEMANTIC SEARCH\n")

query = "Artificial Intelligence in healthcare"

results = collection.query(
    query_texts=[query],
    n_results=5
)


# ============================================
# DISPLAY RESULTS
# ============================================

print("\nTOP SEARCH RESULTS\n")

for i in range(len(results["documents"][0])):

    print(f"\n========== RESULT {i+1} ==========\n")

    print("TITLE:\n")

    print(results["metadatas"][0][i]["title"])

    print("\nURL:\n")

    print(results["metadatas"][0][i]["url"])

    print("\nCONTENT PREVIEW:\n")

    print(results["documents"][0][i][:500])

    print("\n")


# ============================================
# VERIFY CSV SHAPE
# ============================================

test_df = pd.read_csv("news_corpus.csv")

print("\nCSV SHAPE CHECK\n")

print(test_df.shape)


# ============================================
# FINAL SUMMARY
# ============================================

print("""

============================================
PIPELINE COMPLETED SUCCESSFULLY
============================================

Tasks Completed:
1. Web Scraping
2. Corpus Creation
3. Word Tokenization
4. Sentence Tokenization
5. Stopword Removal
6. Embedding Generation
7. Vector Database Storage
8. Semantic Similarity Search

Generated Files:
- corpus.json
- news_corpus.csv
- embeddings.npy

============================================

""")



STEP 1 — SCRAPING NPR NEWS ARTICLES



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\patha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\patha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\patha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Total Links Found: 28
Scraped: TSA's new 'Gold+' program looks to increase private security screening at airports
Scraped: Even as anxieties grow under Trump  these swing voters aren't ready to back Democrats
Scraped: As floods get worse  Britain tries a new solution: beavers
Scraped: COVID is shaping Americans' reaction to Ebola and hantavirus
Scraped: A guide to converting your lawn into a wildlife friendly garden
Scraped: Spencer Pratt is 'winning the internet ' but can he become mayor of Los Angeles?
Scraped: The Education Department is hiring — while it's being dismantled
Scraped: Prediction markets are making a 150-year comeback
Scraped: Officers who defended Capitol from rioters sue to block payouts from fund
Scraped: Ex-prosecutor charged with sending to herself report on Trump classified probe
Scraped: Netanyahu scolds Israeli security minister for videos taunting flotilla activists
Scraped: Ebola fears surge on the ground in Congo over rapid spread of a rare type
Scraped: Lib

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Shape: (17, 384)
Embeddings saved as embeddings.npy

SAMPLE EMBEDDING VECTOR

[-0.04668274 -0.01904898 -0.04193655  0.04899545  0.04937891  0.02382103
  0.15384713 -0.03015106  0.00727269  0.00953192 -0.00815976  0.11445042
 -0.03389861  0.06283335  0.00676613 -0.05487895  0.09991901 -0.07280459
 -0.03887556 -0.01374159]

STEP 5 — STORING IN CHROMADB

Documents stored successfully.

STEP 6 — SEMANTIC SEARCH


TOP SEARCH RESULTS


========== RESULT 1 ==========

TITLE:

Advice for 2026 commencement speakers: Don't bring up AI

URL:

https://www.npr.org/2026/05/20/nx-s1-5822419/ai-colleges-commencement-booing

CONTENT PREVIEW:

Real estate executive Gloria Caulfield (left) was the graduation speaker at University of Central Florida and Big Machine Records CEO Scott Borchetta spoke at Middle Tennessee State University's graduation. Both speakers were booed by students when they brought up artificial intelligence.University of Central Florida and Middle Tennessee State University